[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/05-full-projects/ml-fullproj-abtesting.ipynb)

# Full Project: A/B Testing Analysis

*AIBits Academy · Machine Learning End To End · Full Project*

294,478 website visitors, a genuine data-integrity cleanup, and a real two-proportion z-test that ends in "not significant" — a result this course deliberately doesn't dress up as a win.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

*Small numeric differences from the lesson page are normal: library versions, random seeds and dataset copies change the last digits. The conclusions should agree.*

## Setup

In [ ]:
# Fetch the lesson's dataset(s) into the working folder
import os, io, zipfile, urllib.request, urllib.parse

DATA_BASE = "https://raw.githubusercontent.com/aimldstejas/aibits-genai-notebooks/main/ml/data/"   # course copies live in the notebooks repo
for f in ['ab_testing.csv']:
    if not os.path.exists(f):
        urllib.request.urlretrieve(DATA_BASE + urllib.parse.quote(f), f)
        print('downloaded', f)

In [ ]:
# Imports used throughout this project
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             confusion_matrix, classification_report, mean_squared_error, mean_absolute_error, r2_score)

**Load the experiment log** (one row per visitor: assigned group, page shown, converted or not).

In [ ]:
df = pd.read_csv('ab_testing.csv')
print(df.shape)
df.head()

> **Business Problem**
>
> A company redesigned its landing page and wants to know whether the new page genuinely converts more visitors than the old one — a decision that should be based on a statistical test, not on which raw number happens to be slightly higher.

> **Dataset**
>
> **294,478 rows** — user ID, timestamp, assigned group (control/treatment), landing page shown (old/new), and whether the visitor converted. [Dataset source →](https://statso.io/a-b-testing-case-study/)

## Step 1 — Data Integrity Check First

Before any statistical test, the group/page assignment needs verifying — control should always see the old page, treatment the new one:

In [ ]:
mismatch = df[((df['group']=='control') & (df['landing_page']=='new_page')) |
              ((df['group']=='treatment') & (df['landing_page']=='old_page'))]
print(f"Mismatched rows: {len(mismatch)}")
print(f"Duplicate user_id rows: {df['user_id'].duplicated().sum()}")

3,893 rows (1.3% of the data) have a group/page mismatch — likely a bug in whatever system randomly assigned visitors to the test, or duplicate sessions logged for the same user across groups. These are dropped, along with any remaining duplicate `user_id`, before any conversion rate is computed — running the test on contaminated assignment data would make the entire result unreliable regardless of which statistical test is used afterward.

## Step 2 — The Two-Proportion Z-Test

With 290,584 clean rows, control and treatment conversion rates:

The text drops the mismatched assignments and any remaining duplicate users before testing. That produces `df_clean`.

In [ ]:
df_clean = df.drop(mismatch.index).drop_duplicates(subset='user_id')
print(f"Clean rows: {len(df_clean):,}")

In [ ]:
from statsmodels.stats.proportion import proportions_ztest

control = df_clean[df_clean['group']=='control']['converted']
treatment = df_clean[df_clean['group']=='treatment']['converted']
print(f"Control:   n={len(control)}   rate={control.mean():.4f}")
print(f"Treatment: n={len(treatment)}   rate={treatment.mean():.4f}")

z_stat, p_val = proportions_ztest([control.sum(), treatment.sum()], [len(control), len(treatment)])
print(f"z={z_stat:.4f}   p={p_val:.4f}")

The 95% confidence interval for the difference (treatment − control) is **[−0.0039, +0.0008]** — it contains zero, and the p-value (0.19) is well above the conventional 0.05 threshold.

## Visualizing the Null Result

Left: the two raw conversion rates, nearly the same height. Right: the treatment−control difference and its 95% CI plotted against a zero line — the whisker straddles zero, which is the entire "not significant" conclusion made visible.

> **⚠ The Honest Conclusion: No Significant Difference**
>
> The new landing page did **not** produce a statistically significant change in conversion rate — if anything, the point estimate is slightly lower (11.88% vs. 12.04%), but that gap is entirely consistent with random sampling noise given the confidence interval spans zero. The correct business conclusion is "we cannot conclude the redesign helped," not a forced narrative about a marginal win or loss. This is deliberately not dressed up as a success story — it's the same "don't report a misleading performance claim" discipline from the Ethics in ML page, applied to an A/B test rather than a model comparison.

## Step 3 — Why This Result Is Still Valuable

A null result at this sample size (290,584 users, plenty of statistical power to detect even a modest true effect) is itself a genuine, actionable finding: it means the redesign most likely doesn't move conversion in either direction by a meaningful amount, which tells the company to look elsewhere for improvement rather than rolling out a change that cost design and engineering effort for no measurable benefit.

## Step 4 — Designing the Next Test Properly: Minimum Detectable Effect

The landing-page test above had the statistical power to trust its null result — but that was only true because ~290K users happened to be available. Before running *any* A/B test, the real design question is: **given the sample size we can realistically get, what size of effect could we even detect?** Running an underpowered test and concluding "no effect" is a completely different (and much weaker) claim than running a well-powered test and concluding the same thing — the first test genuinely couldn't have found a real effect even if one existed.

> **Second Example — Swiggy Checkout Messaging Test**
>
> Swiggy's product team wants to test whether a new checkout-page message ("Free delivery on your next 3 orders!") improves order-completion rate above the current 6% baseline. Before running the test, they need to know: how many users does this experiment actually need?

$$\mathrm{MDE} = (t_\alpha + t_{1-\beta}) \cdot \sqrt{\dfrac{\mathrm{Var}(Y)}{N\cdot P\cdot(1-P)}}$$

where N is total sample size, P is the fraction assigned to treatment (0.5 is optimal), and Var(Y) is the variance of the outcome — for a conversion-rate metric, a Bernoulli variable, this is simply `baseline × (1−baseline)`. Larger experiments detect smaller true effects; noisier outcomes require larger experiments to detect the same effect.

In [ ]:
from scipy import stats

def compute_mde(sample_size, var_outcome, size=0.05, power=0.85):
    dof = sample_size - 1
    t_alpha = stats.t.ppf(1-size, dof)
    t_ombeta = stats.t.ppf(power, dof)
    p = 0.5
    den = sample_size*p*(1-p)
    return (t_alpha + t_ombeta) * np.sqrt(var_outcome/den)

baseline = 0.06
var_outcome = baseline*(1-baseline)

for n in [1_000, 10_000, 100_000, 1_000_000]:
    mde = compute_mde(n, var_outcome)
    print(f"N={n:>9,}: MDE={mde:.5f}  -> min detectable rate = {baseline+mde:.2%}")

With only 1,000 users, the test can only detect an implausibly large jump — from 6% to over 10% conversion. A checkout message that genuinely lifted conversion by a modest, business-realistic 1 percentage point (6%→7%) would appear as "no significant effect" purely from lack of power, not because it didn't work. Inverting the formula to solve for required sample size given a target MDE of 1 percentage point:

In [ ]:
def compute_sample_size(mde, var_outcome, data_size, size=0.05, power=0.85):
    dof = data_size - 1
    t_alpha = stats.t.ppf(1-size, dof)
    t_ombeta = stats.t.ppf(power, dof)
    p = 0.5
    return var_outcome/(p*(1-p)) * ((t_alpha+t_ombeta)**2/mde**2)

n_needed = compute_sample_size(mde=0.01, var_outcome=0.06*0.94, data_size=100_000)
print(f"Required total sample size: {n_needed:.0f}")

Swiggy's team needs roughly **16,200 total users** (about 8,100 per arm) to reliably detect a 6%→7% conversion lift at 5% significance and 85% power. If daily checkout traffic is, say, 5,000 users, that's a ~4-day test — a concrete, schedulable answer the product team can act on before the experiment even starts, rather than discovering after two disappointing days that the test was never going to be conclusive either way.

> **⚠ Statistical Significance ≠ Business Significance**
>
> MDE cuts both ways. With Swiggy's full daily active user base (crores of users), the test could detect a lift as small as 0.01 percentage points — technically "significant" but likely worthless from a business standpoint. Setting the target MDE should start from "what size of lift would actually be worth shipping," not from "what's the smallest effect we could possibly measure." The two questions in this Step 4 — "is this test well-powered?" and "is the effect we're chasing even worth chasing?" — both have to be answered before a single user sees the new checkout message, not after.

## Key Business Takeaways

- 1.3% of rows had a group/page assignment mismatch — a reminder to audit test-assignment integrity before trusting any A/B test's headline numbers.
- With ~290K clean rows, this test had strong statistical power, making the null result trustworthy rather than merely "not enough data to tell."
- A null result is a real, useful answer — it tells the company not to ship the redesign expecting a conversion lift, redirecting effort to other hypotheses instead.
- Minimum Detectable Effect (MDE) flips the whole exercise around: before running a test, it tells you the smallest true effect your planned sample size could even detect — roughly 16,200 total users were needed to reliably catch a realistic 6%→7% conversion lift in the Swiggy checkout example.

## Practice Questions

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · Conversion rate per group

From `df_clean`, store the conversion rate of each `group` in the dict `rates` (keys `"control"` and `"treatment"`).

In [ ]:
rates = {}   # TODO


In [ ]:
try:
    check("control", abs(rates["control"] - 0.1204) < 5e-4)
    check("treatment", abs(rates["treatment"] - 0.1188) < 5e-4)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
rates = df_clean.groupby("group")["converted"].mean().to_dict()

```

</details>

### Exercise 2 · Medium · A confidence interval for the difference

Using the normal approximation, compute the 95% CI for `treatment - control` conversion: `diff ± 1.96·sqrt(p1(1-p1)/n1 + p2(1-p2)/n2)`. Store it as `(lo, hi)` in `ci`. Since the interval contains 0, the lesson calls the result not significant.

In [ ]:
ci = None   # TODO


In [ ]:
try:
    check("lesson says about [-0.0039, +0.0008]", abs(ci[0] + 0.0039) < 4e-4 and abs(ci[1] - 0.0008) < 4e-4)
    check("contains zero", ci[0] < 0 < ci[1])
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
c = df_clean[df_clean.group == "control"]["converted"]; t = df_clean[df_clean.group == "treatment"]["converted"]
p1, p2, n1, n2 = c.mean(), t.mean(), len(c), len(t)
se = np.sqrt(p1 * (1 - p1) / n1 + p2 * (1 - p2) / n2)
ci = ((p2 - p1) - 1.96 * se, (p2 - p1) + 1.96 * se)

```

</details>

### Exercise 3 · Stretch · How many visitors per arm?

To detect a lift from 6% to 7% with 80% power at a 5% two-sided significance level, use `statsmodels`' `NormalIndPower` with Cohen's `proportion_effectsize(0.07, 0.06)`. Store the visitors needed **per arm** (rounded up) in `n_per_arm`.

In [ ]:
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize
n_per_arm = None   # TODO


In [ ]:
try:
    check("between 8,000 and 12,000 per arm", 8000 < n_per_arm < 12000)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize
h = proportion_effectsize(0.07, 0.06)
n_per_arm = int(np.ceil(NormalIndPower().solve_power(effect_size=h, alpha=0.05, power=0.80, ratio=1.0, alternative="two-sided")))

```

A test that is too small cannot detect a realistic effect - the power calculation is done *before* the experiment.

</details>

---
*Back to the course: **Machine Learning End To End → Full Project: A/B Testing Analysis**.*